# GKT

Student-blocked nested evaluation of GKT on two binary prediction tasks:

- **First attempt:** `firstattemptcorrect`
- **After feedback:** `eventualcorrect`

Run the setup cell, then either task cell. Each task uses its own fixed outer-fold file and output directory. Hyperparameters and the final training epoch count are selected exclusively from inner validation folds.


In [ ]:
from pathlib import Path
import pandas as pd

# Point this to the interaction-level FeedBook export.
DATA_PATH = Path("data/feedbook_interactions.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}. Update DATA_PATH before running."
    )

logging_data = pd.read_csv(DATA_PATH)
print(f"Loaded {len(logging_data):,} interactions from {DATA_PATH}")


## First-attempt prediction


In [ ]:
"""GKT evaluation with fixed outer folds and crash-safe OOF saving.

Designed for a RunPod/notebook workflow where ``logging_data`` already exists.
To run as a standalone script, replace the marked data-loading block below.

Final training uses an epoch count selected from inner validation folds.
The outer test fold is evaluated once after training.
"""

import gc
import json
import logging
import math
import os
import random
import sys
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from optuna.samplers import GridSampler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold
from torch.utils.data import DataLoader, Dataset


# -----------------------------------------------------------------------------
# Self-contained GKT implementation (adapted from the pyKT GKT implementation)
# -----------------------------------------------------------------------------
class MLP(nn.Module):
    """Two-layer MLP used by GKT for self and neighbour transformations."""

    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.0, bias=True):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim, bias=bias)
        self.fc2 = nn.Linear(hidden_dim, output_dim, bias=bias)
        self.norm = nn.BatchNorm1d(output_dim)
        self.dropout = dropout
        self.output_dim = output_dim
        self.reset_parameters()

    def reset_parameters(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_normal_(module.weight)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0.1)
            elif isinstance(module, nn.BatchNorm1d):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def apply_batch_norm(self, inputs):
        # BatchNorm cannot estimate variance from one feature vector.
        if inputs.numel() == 0 or inputs.numel() == self.output_dim:
            return inputs
        if inputs.ndim == 3:
            original_shape = inputs.shape
            normalized = self.norm(inputs.reshape(-1, original_shape[-1]))
            return normalized.reshape(original_shape)
        return self.norm(inputs)

    def forward(self, inputs):
        outputs = F.relu(self.fc1(inputs))
        outputs = F.dropout(outputs, p=self.dropout, training=self.training)
        outputs = F.relu(self.fc2(outputs))
        return self.apply_batch_norm(outputs)


class EraseAddGate(nn.Module):
    """Erase/add gate used to update every concept state in GKT."""

    def __init__(self, feature_dim, num_c, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(num_c))
        self.erase = nn.Linear(feature_dim, feature_dim, bias=bias)
        self.add = nn.Linear(feature_dim, feature_dim, bias=bias)
        self.reset_parameters()

    def reset_parameters(self):
        bound = 1.0 / math.sqrt(self.weight.size(0))
        nn.init.uniform_(self.weight, -bound, bound)

    def forward(self, inputs):
        erase_gate = torch.sigmoid(self.erase(inputs))
        concept_weight = self.weight.view(1, -1, 1)
        erased = inputs - concept_weight * erase_gate * inputs
        added = torch.tanh(self.add(inputs))
        return erased + concept_weight * added


class GKT(nn.Module):
    """Graph-based Knowledge Tracing with dense or transition adjacency."""

    def __init__(
        self,
        num_c,
        hidden_dim,
        emb_size,
        graph_type="dense",
        graph=None,
        dropout=0.5,
        emb_type="qid",
        emb_path="",
        bias=True,
    ):
        super().__init__()
        if graph is None:
            raise ValueError("GKT requires an adjacency graph tensor.")
        if tuple(graph.shape) != (num_c, num_c):
            raise ValueError(
                f"Graph shape must be {(num_c, num_c)}, got {tuple(graph.shape)}."
            )

        self.model_name = "gkt"
        self.num_c = num_c
        self.hidden_dim = hidden_dim
        self.emb_size = emb_size
        self.res_len = 2
        self.graph_type = graph_type
        self.emb_type = emb_type
        self.emb_path = emb_path

        # Fixed graph stored with the model and moved automatically by .to(device).
        self.register_buffer("graph", graph.detach().clone().float())

        # Fixed one-hot lookup tables are buffers, not trainable parameters.
        self.register_buffer("one_hot_feat", torch.eye(self.res_len * self.num_c))
        self.register_buffer(
            "one_hot_q",
            torch.cat(
                [torch.eye(self.num_c), torch.zeros(1, self.num_c)], dim=0
            ),
        )

        if not emb_type.startswith("qid"):
            raise ValueError("This self-contained experiment supports emb_type='qid'.")

        self.interaction_emb = nn.Embedding(self.res_len * num_c, emb_size)
        # Final row (index num_c) is the internal padding embedding.
        self.emb_c = nn.Embedding(num_c + 1, emb_size, padding_idx=num_c)

        mlp_input_dim = hidden_dim + emb_size
        self.f_self = MLP(
            mlp_input_dim, hidden_dim, hidden_dim, dropout=dropout, bias=bias
        )
        self.f_neighbor_list = nn.ModuleList(
            [
                MLP(
                    2 * mlp_input_dim,
                    hidden_dim,
                    hidden_dim,
                    dropout=dropout,
                    bias=bias,
                ),
                MLP(
                    2 * mlp_input_dim,
                    hidden_dim,
                    hidden_dim,
                    dropout=dropout,
                    bias=bias,
                ),
            ]
        )
        self.erase_add_gate = EraseAddGate(hidden_dim, num_c, bias=bias)
        self.gru = nn.GRUCell(hidden_dim, hidden_dim, bias=bias)
        self.predict_layer = nn.Linear(hidden_dim, 1, bias=bias)

    def _aggregate(self, interaction_t, question_t, hidden_t):
        batch_size = question_t.size(0)
        valid = question_t != -1

        interaction_indices = torch.arange(
            self.res_len * self.num_c, device=question_t.device
        )
        all_interaction_embeddings = self.interaction_emb(interaction_indices)
        selected_one_hot = F.embedding(
            interaction_t[valid].long(), self.one_hot_feat
        )
        response_embeddings = selected_one_hot @ all_interaction_embeddings

        # Invalid/padded rows use the final zero padding embedding.
        concept_indices = torch.full(
            (batch_size, self.num_c),
            self.num_c,
            dtype=torch.long,
            device=question_t.device,
        )
        concept_indices[valid] = torch.arange(
            self.num_c, device=question_t.device
        )
        concept_embeddings = self.emb_c(concept_indices)

        valid_count = int(valid.sum().item())
        if valid_count:
            valid_rows = torch.arange(valid_count, device=question_t.device)
            concept_embeddings[valid, question_t[valid].long()] = response_embeddings

        return torch.cat([hidden_t, concept_embeddings], dim=-1)

    def _aggregate_neighbors(self, temporary_hidden, question_t):
        valid = question_t != -1
        valid_questions = question_t[valid].long()
        valid_temporary = temporary_hidden[valid]
        valid_count = valid_temporary.size(0)

        # Start from the previous hidden component for padded rows.
        messages = temporary_hidden[:, :, : self.hidden_dim].clone()
        if valid_count == 0:
            return messages

        valid_rows = torch.arange(valid_count, device=question_t.device)
        self_hidden = valid_temporary[valid_rows, valid_questions]
        self_features = self.f_self(self_hidden)

        expanded_self = self_hidden.unsqueeze(1).expand(-1, self.num_c, -1)
        neighbor_input = torch.cat([expanded_self, valid_temporary], dim=-1)

        outgoing = self.graph[valid_questions, :].unsqueeze(-1)
        incoming = self.graph[:, valid_questions].transpose(0, 1).unsqueeze(-1)
        neighbor_features = (
            outgoing * self.f_neighbor_list[0](neighbor_input)
            + incoming * self.f_neighbor_list[1](neighbor_input)
        )
        neighbor_features[valid_rows, valid_questions] = self_features
        messages[valid] = neighbor_features
        return messages

    def _update(self, temporary_hidden, hidden_t, question_t):
        valid = question_t != -1
        message_next = self._aggregate_neighbors(temporary_hidden, question_t)
        hidden_next = hidden_t.clone()
        if not valid.any():
            return hidden_next

        gated_messages = self.erase_add_gate(message_next[valid])
        updated = self.gru(
            gated_messages.reshape(-1, self.hidden_dim),
            hidden_t[valid].reshape(-1, self.hidden_dim),
        )
        hidden_next[valid] = updated.reshape(-1, self.num_c, self.hidden_dim)
        return hidden_next

    def _predict_all_concepts(self, hidden_next):
        return torch.sigmoid(self.predict_layer(hidden_next).squeeze(-1))

    def _select_next_prediction(self, all_predictions, next_question):
        # Map external -1 padding to the extra all-zero row in one_hot_q.
        safe_next_question = torch.where(
            next_question != -1,
            next_question,
            torch.full_like(next_question, self.num_c),
        )
        next_one_hot = F.embedding(safe_next_question.long(), self.one_hot_q)
        return (all_predictions * next_one_hot).sum(dim=1)

    def forward(self, questions, responses):
        if questions.shape != responses.shape:
            raise ValueError("questions and responses must have identical shapes.")

        interactions = questions * self.res_len + responses
        batch_size, sequence_length = questions.shape
        hidden_t = torch.zeros(
            batch_size,
            self.num_c,
            self.hidden_dim,
            device=questions.device,
        )

        predictions = []
        for time_index in range(sequence_length - 1):
            question_t = questions[:, time_index]
            interaction_t = interactions[:, time_index]
            temporary_hidden = self._aggregate(
                interaction_t, question_t, hidden_t
            )
            hidden_t = self._update(temporary_hidden, hidden_t, question_t)
            all_predictions = self._predict_all_concepts(hidden_t)
            predictions.append(
                self._select_next_prediction(
                    all_predictions, questions[:, time_index + 1]
                )
            )

        return torch.stack(predictions, dim=1)


# -----------------------------------------------------------------------------
# Configuration: change only this section for another task/KC
# -----------------------------------------------------------------------------
SEED = 42
N_SPLITS = 3
SEQ_LEN = 20  # GKT is memory-heavy; keep identical across compared GKT runs.
BATCH_SIZE = 8
MAX_EPOCHS = 100
PATIENCE = 10
NUM_WORKERS = 0  # Use the main process; avoids DataLoader worker/file-descriptor exhaustion.
GRAPH_TYPE = "transition"

MODEL_NAME = "GKT"
KC_NAME = "exerciseid"  # Used only in filenames/OOF metadata.
QUESTION_COL = "KC (exerciseid)"       # Change to the actual KC column if needed.
CORRECT_COL = "firstattemptcorrect"
STUDENT_COL = "Anon Student Id"
TIMESTAMP_COL = "First Transaction Time"

# For the first-attempt task, normally change these three values:
# CORRECT_COL = "firstattemptcorrect"
# FIXED_FOLD_FILE = "data/fixed_outer_folds.csv"
# OUTPUT_DIR = Path("GKT_Optimized_FA")
FIXED_FOLD_FILE = "data/fixed_outer_folds.csv"
OUTPUT_DIR = Path("GKT_FA")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

PARAM_SEARCH_SPACE = {
    "hidden_dim": [32, 64, 128],
    "learning_rate": [1e-2, 1e-3, 1e-4],
    "dropout": [0.1, 0.3, 0.5],
}


# -----------------------------------------------------------------------------
# Reproducibility, device, logging, and atomic saving
# -----------------------------------------------------------------------------
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = device.type == "cuda"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger("gkt_experiment")
logger.setLevel(logging.INFO)
logger.handlers.clear()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(formatter)
file_handler = logging.FileHandler(OUTPUT_DIR / "run.log", mode="a")
file_handler.setFormatter(formatter)
logger.addHandler(stream_handler)
logger.addHandler(file_handler)


def atomic_csv(df, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)


def atomic_json(payload, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, default=str)
    os.replace(tmp, path)


def atomic_torch_save(payload, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, tmp)
    os.replace(tmp, path)


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def rmse_score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def safe_auc(y_true, y_pred):
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, y_pred))


def metrics_dict(y_true, y_prob):
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob > 0.5).astype(int)
    return {
        "AUC": safe_auc(y_true, y_prob),
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "RMSE": rmse_score(y_true, y_prob),
        "MAE": float(mean_absolute_error(y_true, y_prob)),
        "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "Recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
    }


# -----------------------------------------------------------------------------
# Data and fixed outer folds
# -----------------------------------------------------------------------------
# Uncomment/adapt this when running as a standalone .py file:
# logging_data = pd.read_csv("your_original_dataset.csv")
if "logging_data" not in globals():
    raise RuntimeError(
        "logging_data is not defined. Load the original dataset before running "
        "this script, or uncomment the pd.read_csv line above."
    )

# row_id must be created on the unfiltered original data for cross-model OOF joins.
logging_data = logging_data.copy().reset_index(drop=True)
if "row_id" not in logging_data.columns:
    logging_data["row_id"] = np.arange(len(logging_data), dtype=np.int64)

required_columns = {"row_id", QUESTION_COL, CORRECT_COL, STUDENT_COL}
missing_columns = required_columns.difference(logging_data.columns)
if missing_columns:
    raise KeyError(f"Missing required columns: {sorted(missing_columns)}")

fixed_fold_path = Path(FIXED_FOLD_FILE)
if fixed_fold_path.exists():
    fixed_folds = pd.read_csv(fixed_fold_path)
    required_fold_columns = {"row_id", "outer_fold"}
    if not required_fold_columns.issubset(fixed_folds.columns):
        raise ValueError(
            f"{FIXED_FOLD_FILE} must contain row_id and outer_fold columns."
        )
    if fixed_folds["row_id"].duplicated().any():
        raise ValueError(f"Duplicate row_id values found in {FIXED_FOLD_FILE}.")
    logging_data = logging_data.drop(columns=["outer_fold"], errors="ignore")
    logging_data = logging_data.merge(
        fixed_folds[["row_id", "outer_fold"]], on="row_id", how="left", validate="one_to_one"
    )
    logger.info("Loaded fixed folds from %s", FIXED_FOLD_FILE)
else:
    base_for_folds = logging_data.dropna(subset=[STUDENT_COL]).copy()
    base_for_folds["outer_fold"] = -1
    splitter = GroupKFold(n_splits=N_SPLITS)
    for fold, (_, test_idx) in enumerate(
        splitter.split(base_for_folds, groups=base_for_folds[STUDENT_COL]), start=1
    ):
        base_for_folds.iloc[
            test_idx, base_for_folds.columns.get_loc("outer_fold")
        ] = fold
    fixed_folds = base_for_folds[["row_id", "outer_fold"]].copy()
    atomic_csv(fixed_folds, fixed_fold_path)
    logging_data = logging_data.merge(
        fixed_folds, on="row_id", how="left", validate="one_to_one"
    )
    logger.info("Saved fixed folds to %s", FIXED_FOLD_FILE)

logging_model = logging_data.dropna(
    subset=[QUESTION_COL, CORRECT_COL, STUDENT_COL, "outer_fold"]
).copy()
logging_model[CORRECT_COL] = logging_model[CORRECT_COL].astype(int)
logging_model["outer_fold"] = logging_model["outer_fold"].astype(int)

if not logging_model[CORRECT_COL].isin([0, 1]).all():
    raise ValueError(f"{CORRECT_COL} must contain only binary values 0 and 1.")

observed_folds = set(logging_model["outer_fold"].unique())
expected_folds = set(range(1, N_SPLITS + 1))
if observed_folds != expected_folds:
    raise ValueError(
        f"Expected outer folds {sorted(expected_folds)}, found {sorted(observed_folds)}."
    )

# Every student must belong to exactly one outer fold.
student_fold_counts = logging_model.groupby(STUDENT_COL)["outer_fold"].nunique()
if (student_fold_counts > 1).any():
    raise ValueError("At least one student occurs in more than one outer fold.")

# Stable chronological order for sequences and transition edges.
sort_columns = [STUDENT_COL]
if TIMESTAMP_COL in logging_model.columns:
    sort_columns.append(TIMESTAMP_COL)
else:
    logger.warning(
        "Timestamp column %r was not found; row_id order will be used within students.",
        TIMESTAMP_COL,
    )
sort_columns.append("row_id")
logging_model = logging_model.sort_values(sort_columns, kind="mergesort").reset_index(drop=True)

# Native pyKT GKT convention: real concepts are 0..N-1 and padding is -1.
all_qids = logging_model[QUESTION_COL].drop_duplicates().tolist()
qid_to_index = {qid: index for index, qid in enumerate(all_qids)}
num_questions = len(qid_to_index)
if num_questions < 2:
    raise ValueError("GKT requires at least two distinct concepts.")

student_id_lookup = (
    logging_model[["row_id", STUDENT_COL]]
    .drop_duplicates(subset=["row_id"])
    .set_index("row_id")[STUDENT_COL]
    .to_dict()
)

logger.info("Device: %s", device)
logger.info("Model: %s | KC: %s | target: %s", MODEL_NAME, KC_NAME, CORRECT_COL)
logger.info("seq_len: %d | num_concepts: %d", SEQ_LEN, num_questions)


# -----------------------------------------------------------------------------
# Correct graph construction: one global KC mapping, training data only
# -----------------------------------------------------------------------------
def build_transition_graph_from_df(df, concept_num, qid_mapping):
    graph = np.zeros((concept_num, concept_num), dtype=np.float32)
    ordered = df.sort_values(sort_columns, kind="mergesort")

    for _, group in ordered.groupby(STUDENT_COL, sort=False):
        sequence = group[QUESTION_COL].map(qid_mapping).dropna().astype(int).to_numpy()
        if len(sequence) < 2:
            continue
        previous = sequence[:-1]
        following = sequence[1:]
        non_self = previous != following
        np.add.at(graph, (previous[non_self], following[non_self]), 1.0)

    row_sums = graph.sum(axis=1, keepdims=True)
    np.divide(graph, row_sums, out=graph, where=row_sums != 0)
    return torch.tensor(graph, dtype=torch.float32)


def build_dense_graph(concept_num):
    graph = np.full(
        (concept_num, concept_num), 1.0 / (concept_num - 1), dtype=np.float32
    )
    np.fill_diagonal(graph, 0.0)
    return torch.tensor(graph, dtype=torch.float32)


def get_gkt_graph(df):
    if GRAPH_TYPE == "transition":
        return build_transition_graph_from_df(df, num_questions, qid_to_index)
    if GRAPH_TYPE == "dense":
        return build_dense_graph(num_questions)
    raise ValueError("GRAPH_TYPE must be 'transition' or 'dense'.")


# -----------------------------------------------------------------------------
# Dataset: keeps row_id so every prediction can be aligned across models
# -----------------------------------------------------------------------------
class KTDataFromLogging(Dataset):
    def __init__(self, df):
        self.samples = []
        ordered = df.sort_values(sort_columns, kind="mergesort").copy()
        ordered["qid_index"] = ordered[QUESTION_COL].map(qid_to_index)
        if ordered["qid_index"].isna().any():
            raise ValueError("A question could not be mapped to the global KC index.")
        ordered["qid_index"] = ordered["qid_index"].astype(int)

        for _, group in ordered.groupby(STUDENT_COL, sort=False):
            q_seq = group["qid_index"].tolist()
            r_seq = group[CORRECT_COL].astype(int).tolist()
            row_seq = group["row_id"].astype(int).tolist()

            # Adjacent windows share one interaction so every target retains context.
            for start in range(0, len(q_seq), SEQ_LEN - 1):
                end = min(start + SEQ_LEN, len(q_seq))

                if end - start < 2:
                    break

                q_chunk = q_seq[start:end]
                r_chunk = r_seq[start:end]
                row_chunk = row_seq[start:end]
                pad_len = SEQ_LEN - len(q_chunk)
                if pad_len:
                    q_chunk += [-1] * pad_len
                    r_chunk += [-1] * pad_len
                    row_chunk += [-1] * pad_len
                self.samples.append(
                    (
                        torch.tensor(q_chunk, dtype=torch.long),
                        torch.tensor(r_chunk, dtype=torch.long),
                        torch.tensor(row_chunk, dtype=torch.long),
                    )
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]


def make_loader(df, shuffle):
    """
    Build a deterministic in-memory DataLoader.

    This dataset is already materialized as tensors in KTDataFromLogging, so
    multiprocessing workers are unnecessary here. Keeping NUM_WORKERS=0 also
    avoids accumulating worker pipes/epoll file descriptors across the many
    short-lived loaders created during nested CV/grid search.
    """
    loader_kwargs = {
        "dataset": KTDataFromLogging(df),
        "batch_size": BATCH_SIZE,
        "shuffle": shuffle,
        "num_workers": NUM_WORKERS,
        "pin_memory": pin_memory,
    }

    # Only enable worker persistence if multiprocessing is explicitly turned
    # back on in a future run. With NUM_WORKERS=0 no worker processes exist.
    if NUM_WORKERS > 0:
        loader_kwargs["persistent_workers"] = False

    return DataLoader(**loader_kwargs)


def make_model(params, graph):
    return GKT(
        num_c=num_questions,
        hidden_dim=params["hidden_dim"],
        emb_size=params["hidden_dim"],
        graph_type=GRAPH_TYPE,
        graph=graph,
        dropout=params["dropout"],
        emb_type="qid",
    ).to(device)


criterion = nn.BCELoss()


def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    batches = 0
    for q_batch, r_batch, _ in loader:
        q_batch = q_batch.to(device, non_blocking=True)
        r_batch = r_batch.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(q_batch, r_batch)  # [batch, seq_len - 1]
        mask = q_batch[:, 1:] != -1
        if not mask.any():
            continue
        loss = criterion(outputs[mask], r_batch[:, 1:].float()[mask])
        loss.backward()
        optimizer.step()
        total_loss += float(loss.item())
        batches += 1
    return total_loss / max(batches, 1)


def predict(model, loader, include_row_ids=False):
    model.eval()
    predictions, labels, row_ids = [], [], []
    with torch.no_grad():
        for q_batch, r_batch, rowid_batch in loader:
            q_batch = q_batch.to(device, non_blocking=True)
            r_batch = r_batch.to(device, non_blocking=True)
            outputs = model(q_batch, r_batch)
            mask = q_batch[:, 1:] != -1
            predictions.extend(outputs[mask].detach().cpu().numpy().tolist())
            labels.extend(r_batch[:, 1:][mask].detach().cpu().numpy().tolist())
            if include_row_ids:
                row_ids.extend(rowid_batch[:, 1:][mask.cpu()].numpy().tolist())
    if include_row_ids:
        return predictions, labels, row_ids
    return predictions, labels


# -----------------------------------------------------------------------------
# Outer CV
# -----------------------------------------------------------------------------
all_fold_frames = []
fold_metric_rows = []
grid_size = int(np.prod([len(values) for values in PARAM_SEARCH_SPACE.values()]))

config_payload = {
    "seed": SEED,
    "n_splits": N_SPLITS,
    "seq_len": SEQ_LEN,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "graph_type": GRAPH_TYPE,
    "model_name": MODEL_NAME,
    "kc_name": KC_NAME,
    "question_col": QUESTION_COL,
    "correct_col": CORRECT_COL,
    "student_col": STUDENT_COL,
    "timestamp_col": TIMESTAMP_COL,
    "fixed_fold_file": FIXED_FOLD_FILE,
    "param_search_space": PARAM_SEARCH_SPACE,
    "num_questions": num_questions,
    # Intentionally matches the legacy protocol used by the comparison models.
    "outer_test_used_during_training": False,
    "epoch_selection": "median_inner_best_epoch",
}
atomic_json(config_payload, OUTPUT_DIR / "run_config.json")

for fold in range(1, N_SPLITS + 1):
    fold_oof_path = OUTPUT_DIR / f"oof_{MODEL_NAME}_{KC_NAME}_fold{fold}.csv"
    fold_result_path = CHECKPOINT_DIR / f"fold_{fold}_results.csv"
    fold_model_path = CHECKPOINT_DIR / f"best_model_fold_{fold}.pth"
    training_state_path = CHECKPOINT_DIR / f"training_state_fold_{fold}.pth"

    # The result CSV is written last and acts as the fold-completion marker.
    if fold_result_path.exists() and fold_oof_path.exists() and fold_model_path.exists():
        logger.info("Fold %d is complete; loading saved OOF and metrics.", fold)
        saved_oof = pd.read_csv(fold_oof_path)
        saved_result = pd.read_csv(fold_result_path)
        all_fold_frames.append(saved_oof)
        fold_metric_rows.append(saved_result.iloc[0].to_dict())
        continue

    logger.info("=== Outer Fold %d/%d ===", fold, N_SPLITS)
    train_val_df = logging_model[logging_model["outer_fold"] != fold].copy()
    test_df = logging_model[logging_model["outer_fold"] == fold].copy()

    overlap = set(train_val_df[STUDENT_COL]).intersection(test_df[STUDENT_COL])
    if overlap:
        raise RuntimeError(f"Student leakage detected in outer fold {fold}.")

    inner_cv = GroupKFold(n_splits=N_SPLITS)
    inner_groups = train_val_df[STUDENT_COL]

    def objective(trial):
        params = {
            "hidden_dim": trial.suggest_categorical(
                "hidden_dim", PARAM_SEARCH_SPACE["hidden_dim"]
            ),
            "learning_rate": trial.suggest_categorical(
                "learning_rate", PARAM_SEARCH_SPACE["learning_rate"]
            ),
            "dropout": trial.suggest_categorical(
                "dropout", PARAM_SEARCH_SPACE["dropout"]
            ),
        }
        inner_aucs = []
        best_epochs = []

        try:
            for inner_fold, (inner_train_idx, inner_val_idx) in enumerate(
                inner_cv.split(train_val_df, groups=inner_groups), start=1
            ):
                inner_train_df = train_val_df.iloc[inner_train_idx].copy()
                inner_val_df = train_val_df.iloc[inner_val_idx].copy()

                # Crucial: the graph sees inner-training data only.
                inner_graph = get_gkt_graph(inner_train_df).to(device)
                train_loader = make_loader(inner_train_df, shuffle=True)
                val_loader = make_loader(inner_val_df, shuffle=False)
                model = make_model(params, inner_graph)
                optimizer = torch.optim.Adam(
                    model.parameters(), lr=params["learning_rate"]
                )

                best_auc = -np.inf
                best_epoch = 1
                no_improve = 0
                for epoch in range(1, MAX_EPOCHS + 1):
                    train_one_epoch(model, train_loader, optimizer)
                    val_preds, val_labels = predict(model, val_loader)
                    val_auc = safe_auc(val_labels, val_preds)
                    if np.isfinite(val_auc) and val_auc > best_auc:
                        best_auc = val_auc
                        best_epoch = epoch
                        no_improve = 0
                    else:
                        no_improve += 1
                    if no_improve >= PATIENCE:
                        break

                if not np.isfinite(best_auc):
                    raise RuntimeError(
                        f"AUC is undefined in outer fold {fold}, inner fold {inner_fold}."
                    )
                inner_aucs.append(float(best_auc))
                best_epochs.append(int(best_epoch))

                del model, optimizer, train_loader, val_loader, inner_graph
                clear_memory()

        except (torch.cuda.OutOfMemoryError, MemoryError) as error:
            logger.warning("Pruning OOM trial with parameters %s", params)
            clear_memory()
            raise optuna.TrialPruned("Out of memory") from error

        recommended_epochs = max(1, int(np.median(best_epochs)))
        trial.set_user_attr("recommended_epochs", recommended_epochs)
        trial.set_user_attr("inner_best_epochs", best_epochs)
        return float(np.mean(inner_aucs))

    storage_path = (CHECKPOINT_DIR / f"optuna_gkt_{KC_NAME}_fold{fold}.db").resolve()
    storage_url = f"sqlite:///{storage_path}"
    study = optuna.create_study(
        study_name=f"{MODEL_NAME}_{KC_NAME}_{CORRECT_COL}_fold{fold}",
        storage=storage_url,
        load_if_exists=True,
        direction="maximize",
        sampler=GridSampler(PARAM_SEARCH_SPACE, seed=SEED),
    )

    # ------------------------------------------------------------------
    # Resume the exhaustive grid safely.
    #
    # GridSampler records failed grid IDs as visited. Therefore, if a trial
    # failed because of an infrastructure problem (for example a DataLoader
    # worker/file-descriptor failure), simply reloading the study can leave
    # that parameter combination without a valid objective value.
    #
    # Re-enqueue any FAILED parameter combination that does not already have
    # a COMPLETE evaluation. This preserves all completed tuning work while
    # ensuring the failed grid point is actually evaluated.
    # ------------------------------------------------------------------
    parameter_names = tuple(PARAM_SEARCH_SPACE.keys())

    def params_key(params):
        if not all(name in params for name in parameter_names):
            return None
        return tuple(params[name] for name in parameter_names)

    complete_keys = {
        params_key(trial.params)
        for trial in study.trials
        if trial.state == optuna.trial.TrialState.COMPLETE
        and params_key(trial.params) is not None
    }

    queued_or_running_keys = {
        params_key(trial.params)
        for trial in study.trials
        if trial.state in {
            optuna.trial.TrialState.WAITING,
            optuna.trial.TrialState.RUNNING,
        }
        and params_key(trial.params) is not None
    }

    for failed_trial in study.trials:
        if failed_trial.state != optuna.trial.TrialState.FAIL:
            continue

        failed_key = params_key(failed_trial.params)
        if failed_key is None:
            continue

        if failed_key in complete_keys or failed_key in queued_or_running_keys:
            continue

        retry_params = {
            name: failed_trial.params[name]
            for name in parameter_names
        }
        study.enqueue_trial(
            retry_params,
            user_attrs={"retry_of_failed_trial": int(failed_trial.number)},
        )
        queued_or_running_keys.add(failed_key)
        logger.info(
            "Re-enqueued failed trial %d with params %s.",
            failed_trial.number,
            retry_params,
        )

    # Count unique grid combinations that already have a usable terminal
    # result. PRUNED trials retain the original behavior and count as done.
    finished_states = {
        optuna.trial.TrialState.COMPLETE,
        optuna.trial.TrialState.PRUNED,
    }
    finished_keys = {
        params_key(trial.params)
        for trial in study.trials
        if trial.state in finished_states
        and params_key(trial.params) is not None
    }

    remaining_trials = max(0, grid_size - len(finished_keys))

    logger.info(
        "Fold %d tuning resume: %d/%d unique grid combinations finished; "
        "%d trial(s) still required.",
        fold,
        len(finished_keys),
        grid_size,
        remaining_trials,
    )

    if remaining_trials:
        study.optimize(
            objective,
            n_trials=remaining_trials,
            show_progress_bar=True,
            gc_after_trial=True,
        )

    best_trial = study.best_trial
    best_params = dict(best_trial.params)
    logger.info(
        "Fold %d best params: %s | tuning AUC: %.6f",
        fold,
        best_params,
        best_trial.value,
    )

    # LEGACY COMPARABILITY PROTOCOL:
    # Retrain on all outer-training students for the epoch count selected
    # exclusively from the inner validation folds.
    final_graph = get_gkt_graph(train_val_df).to(device)
    train_loader = make_loader(train_val_df, shuffle=True)
    test_loader = make_loader(test_df, shuffle=False)
    best_model = make_model(best_params, final_graph)
    optimizer = torch.optim.Adam(
        best_model.parameters(), lr=best_params["learning_rate"]
    )

    final_epochs = int(best_trial.user_attrs["recommended_epochs"])
    start_epoch = 1
    completed_epoch = 0

    if training_state_path.exists():
        recovery = torch.load(
            training_state_path, map_location=device, weights_only=False
        )
        if (
            recovery.get("best_params") == best_params
            and int(recovery.get("final_epochs", -1)) == final_epochs
        ):
            best_model.load_state_dict(recovery["model_state_dict"])
            optimizer.load_state_dict(recovery["optimizer_state_dict"])
            start_epoch = int(recovery["completed_epoch"]) + 1
            completed_epoch = int(recovery["completed_epoch"])
            random.setstate(recovery["python_rng_state"])
            np.random.set_state(recovery["numpy_rng_state"])
            torch.set_rng_state(recovery["torch_rng_state"])
            if torch.cuda.is_available() and recovery.get("cuda_rng_state_all") is not None:
                torch.cuda.set_rng_state_all(recovery["cuda_rng_state_all"])
            logger.info("Resuming fold %d final training at epoch %d.", fold, start_epoch)

    for epoch in range(start_epoch, final_epochs + 1):
        epoch_loss = train_one_epoch(best_model, train_loader, optimizer)
        completed_epoch = epoch
        atomic_torch_save(
            {
                "completed_epoch": epoch,
                "final_epochs": final_epochs,
                "best_params": best_params,
                "model_state_dict": best_model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "python_rng_state": random.getstate(),
                "numpy_rng_state": np.random.get_state(),
                "torch_rng_state": torch.get_rng_state(),
                "cuda_rng_state_all": (
                    torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
                ),
            },
            training_state_path,
        )
        logger.info(
            "Fold %d final training epoch %d/%d | loss %.6f",
            fold, epoch, final_epochs, epoch_loss,
        )

    # Evaluate the untouched outer test fold once.
    fold_preds, fold_labels, fold_row_ids = predict(
        best_model, test_loader, include_row_ids=True
    )
    if not fold_labels:
        raise RuntimeError(f"No valid test predictions were produced for fold {fold}.")
    if len(fold_row_ids) != len(set(fold_row_ids)):
        raise RuntimeError(f"Duplicate row_id predictions found in fold {fold}.")

    fold_metrics = metrics_dict(fold_labels, fold_preds)
    logger.info(
        "Fold %d test | AUC %.6f | Accuracy %.6f | RMSE %.6f | MAE %.6f | "
        "Precision %.6f | Recall %.6f | F1 %.6f",
        fold,
        fold_metrics["AUC"],
        fold_metrics["Accuracy"],
        fold_metrics["RMSE"],
        fold_metrics["MAE"],
        fold_metrics["Precision"],
        fold_metrics["Recall"],
        fold_metrics["F1"],
    )

    fold_df = pd.DataFrame(
        {
            "row_id": fold_row_ids,
            "student_id": [student_id_lookup[row_id] for row_id in fold_row_ids],
            "fold": fold,
            "y_true": fold_labels,
            "y_pred": fold_preds,
            "model_name": MODEL_NAME,
            "kc_name": KC_NAME,
        }
    ).sort_values("row_id").reset_index(drop=True)

    result_row = {
        "Fold": fold,
        "Tuning_AUC": float(best_trial.value),
        **fold_metrics,
        "Final_Epochs": final_epochs,
        "Best_Params": json.dumps(best_params, sort_keys=True),
    }

    # Crash-safe order: OOF -> model -> completion marker (result CSV).
    atomic_csv(fold_df, fold_oof_path)
    atomic_torch_save(
        {
            "model_state_dict": best_model.state_dict(),
            "best_params": best_params,
            "final_epochs": final_epochs,
            "qid_to_index": qid_to_index,
            "config": config_payload,
        },
        fold_model_path,
    )
    atomic_csv(pd.DataFrame([result_row]), fold_result_path)
    logger.info("Saved fold %d model, metrics, and OOF predictions.", fold)

    all_fold_frames.append(fold_df)
    fold_metric_rows.append(result_row)
    del best_model, optimizer, train_loader, test_loader, final_graph
    clear_memory()


# -----------------------------------------------------------------------------
# Combined OOF and final pooled/fold-averaged results (also works after resume)
# -----------------------------------------------------------------------------
if len(all_fold_frames) != N_SPLITS:
    raise RuntimeError(
        f"Expected {N_SPLITS} completed fold OOF files, found {len(all_fold_frames)}."
    )

oof_df = pd.concat(all_fold_frames, ignore_index=True).sort_values("row_id").reset_index(drop=True)
if oof_df["row_id"].duplicated().any():
    raise RuntimeError("Duplicate row_id values found across combined OOF predictions.")

combined_oof_path = OUTPUT_DIR / f"oof_{MODEL_NAME}_{KC_NAME}_all.csv"
atomic_csv(oof_df, combined_oof_path)

fold_results_df = pd.DataFrame(fold_metric_rows).sort_values("Fold").reset_index(drop=True)
atomic_csv(fold_results_df, OUTPUT_DIR / "fold_results_all.csv")

metric_names = ["AUC", "Accuracy", "RMSE", "MAE", "Precision", "Recall", "F1"]
pooled = metrics_dict(oof_df["y_true"], oof_df["y_pred"])
summary_rows = []
for metric in metric_names:
    values = pd.to_numeric(fold_results_df[metric], errors="coerce").to_numpy(dtype=float)
    values = values[np.isfinite(values)]
    summary_rows.append(
        {
            "Metric": "F1 Score" if metric == "F1" else metric,
            "Pooled Score": pooled[metric],
            "Average Score": float(np.mean(values)),
            "Std": float(np.std(values, ddof=1)) if len(values) > 1 else np.nan,
        }
    )

summary_df = pd.DataFrame(summary_rows)
atomic_csv(summary_df, OUTPUT_DIR / "final_metrics.csv")

logger.info("Saved combined OOF predictions to %s", combined_oof_path)
logger.info("=== Final Evaluation Across All Folds ===")
logger.info("Metric       | Pooled Score | Average Score ± Std")
logger.info("-------------|--------------|---------------------")
for row in summary_rows:
    logger.info(
        "%-12s | %.6f     | %.6f ± %.6f",
        row["Metric"],
        row["Pooled Score"],
        row["Average Score"],
        row["Std"],
    )

## After-feedback prediction


In [ ]:
"""GKT evaluation with fixed outer folds and crash-safe OOF saving.

Designed for a RunPod/notebook workflow where ``logging_data`` already exists.
To run as a standalone script, replace the marked data-loading block below.

Final training uses an epoch count selected from inner validation folds.
The outer test fold is evaluated once after training.
"""

import gc
import json
import logging
import math
import os
import random
import sys
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from optuna.samplers import GridSampler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold
from torch.utils.data import DataLoader, Dataset


# -----------------------------------------------------------------------------
# Self-contained GKT implementation (adapted from the pyKT GKT implementation)
# -----------------------------------------------------------------------------
class MLP(nn.Module):
    """Two-layer MLP used by GKT for self and neighbour transformations."""

    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.0, bias=True):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim, bias=bias)
        self.fc2 = nn.Linear(hidden_dim, output_dim, bias=bias)
        self.norm = nn.BatchNorm1d(output_dim)
        self.dropout = dropout
        self.output_dim = output_dim
        self.reset_parameters()

    def reset_parameters(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_normal_(module.weight)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0.1)
            elif isinstance(module, nn.BatchNorm1d):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def apply_batch_norm(self, inputs):
        # BatchNorm cannot estimate variance from one feature vector.
        if inputs.numel() == 0 or inputs.numel() == self.output_dim:
            return inputs
        if inputs.ndim == 3:
            original_shape = inputs.shape
            normalized = self.norm(inputs.reshape(-1, original_shape[-1]))
            return normalized.reshape(original_shape)
        return self.norm(inputs)

    def forward(self, inputs):
        outputs = F.relu(self.fc1(inputs))
        outputs = F.dropout(outputs, p=self.dropout, training=self.training)
        outputs = F.relu(self.fc2(outputs))
        return self.apply_batch_norm(outputs)


class EraseAddGate(nn.Module):
    """Erase/add gate used to update every concept state in GKT."""

    def __init__(self, feature_dim, num_c, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(num_c))
        self.erase = nn.Linear(feature_dim, feature_dim, bias=bias)
        self.add = nn.Linear(feature_dim, feature_dim, bias=bias)
        self.reset_parameters()

    def reset_parameters(self):
        bound = 1.0 / math.sqrt(self.weight.size(0))
        nn.init.uniform_(self.weight, -bound, bound)

    def forward(self, inputs):
        erase_gate = torch.sigmoid(self.erase(inputs))
        concept_weight = self.weight.view(1, -1, 1)
        erased = inputs - concept_weight * erase_gate * inputs
        added = torch.tanh(self.add(inputs))
        return erased + concept_weight * added


class GKT(nn.Module):
    """Graph-based Knowledge Tracing with dense or transition adjacency."""

    def __init__(
        self,
        num_c,
        hidden_dim,
        emb_size,
        graph_type="dense",
        graph=None,
        dropout=0.5,
        emb_type="qid",
        emb_path="",
        bias=True,
    ):
        super().__init__()
        if graph is None:
            raise ValueError("GKT requires an adjacency graph tensor.")
        if tuple(graph.shape) != (num_c, num_c):
            raise ValueError(
                f"Graph shape must be {(num_c, num_c)}, got {tuple(graph.shape)}."
            )

        self.model_name = "gkt"
        self.num_c = num_c
        self.hidden_dim = hidden_dim
        self.emb_size = emb_size
        self.res_len = 2
        self.graph_type = graph_type
        self.emb_type = emb_type
        self.emb_path = emb_path

        # Fixed graph stored with the model and moved automatically by .to(device).
        self.register_buffer("graph", graph.detach().clone().float())

        # Fixed one-hot lookup tables are buffers, not trainable parameters.
        self.register_buffer("one_hot_feat", torch.eye(self.res_len * self.num_c))
        self.register_buffer(
            "one_hot_q",
            torch.cat(
                [torch.eye(self.num_c), torch.zeros(1, self.num_c)], dim=0
            ),
        )

        if not emb_type.startswith("qid"):
            raise ValueError("This self-contained experiment supports emb_type='qid'.")

        self.interaction_emb = nn.Embedding(self.res_len * num_c, emb_size)
        # Final row (index num_c) is the internal padding embedding.
        self.emb_c = nn.Embedding(num_c + 1, emb_size, padding_idx=num_c)

        mlp_input_dim = hidden_dim + emb_size
        self.f_self = MLP(
            mlp_input_dim, hidden_dim, hidden_dim, dropout=dropout, bias=bias
        )
        self.f_neighbor_list = nn.ModuleList(
            [
                MLP(
                    2 * mlp_input_dim,
                    hidden_dim,
                    hidden_dim,
                    dropout=dropout,
                    bias=bias,
                ),
                MLP(
                    2 * mlp_input_dim,
                    hidden_dim,
                    hidden_dim,
                    dropout=dropout,
                    bias=bias,
                ),
            ]
        )
        self.erase_add_gate = EraseAddGate(hidden_dim, num_c, bias=bias)
        self.gru = nn.GRUCell(hidden_dim, hidden_dim, bias=bias)
        self.predict_layer = nn.Linear(hidden_dim, 1, bias=bias)

    def _aggregate(self, interaction_t, question_t, hidden_t):
        batch_size = question_t.size(0)
        valid = question_t != -1

        interaction_indices = torch.arange(
            self.res_len * self.num_c, device=question_t.device
        )
        all_interaction_embeddings = self.interaction_emb(interaction_indices)
        selected_one_hot = F.embedding(
            interaction_t[valid].long(), self.one_hot_feat
        )
        response_embeddings = selected_one_hot @ all_interaction_embeddings

        # Invalid/padded rows use the final zero padding embedding.
        concept_indices = torch.full(
            (batch_size, self.num_c),
            self.num_c,
            dtype=torch.long,
            device=question_t.device,
        )
        concept_indices[valid] = torch.arange(
            self.num_c, device=question_t.device
        )
        concept_embeddings = self.emb_c(concept_indices)

        valid_count = int(valid.sum().item())
        if valid_count:
            valid_rows = torch.arange(valid_count, device=question_t.device)
            concept_embeddings[valid, question_t[valid].long()] = response_embeddings

        return torch.cat([hidden_t, concept_embeddings], dim=-1)

    def _aggregate_neighbors(self, temporary_hidden, question_t):
        valid = question_t != -1
        valid_questions = question_t[valid].long()
        valid_temporary = temporary_hidden[valid]
        valid_count = valid_temporary.size(0)

        # Start from the previous hidden component for padded rows.
        messages = temporary_hidden[:, :, : self.hidden_dim].clone()
        if valid_count == 0:
            return messages

        valid_rows = torch.arange(valid_count, device=question_t.device)
        self_hidden = valid_temporary[valid_rows, valid_questions]
        self_features = self.f_self(self_hidden)

        expanded_self = self_hidden.unsqueeze(1).expand(-1, self.num_c, -1)
        neighbor_input = torch.cat([expanded_self, valid_temporary], dim=-1)

        outgoing = self.graph[valid_questions, :].unsqueeze(-1)
        incoming = self.graph[:, valid_questions].transpose(0, 1).unsqueeze(-1)
        neighbor_features = (
            outgoing * self.f_neighbor_list[0](neighbor_input)
            + incoming * self.f_neighbor_list[1](neighbor_input)
        )
        neighbor_features[valid_rows, valid_questions] = self_features
        messages[valid] = neighbor_features
        return messages

    def _update(self, temporary_hidden, hidden_t, question_t):
        valid = question_t != -1
        message_next = self._aggregate_neighbors(temporary_hidden, question_t)
        hidden_next = hidden_t.clone()
        if not valid.any():
            return hidden_next

        gated_messages = self.erase_add_gate(message_next[valid])
        updated = self.gru(
            gated_messages.reshape(-1, self.hidden_dim),
            hidden_t[valid].reshape(-1, self.hidden_dim),
        )
        hidden_next[valid] = updated.reshape(-1, self.num_c, self.hidden_dim)
        return hidden_next

    def _predict_all_concepts(self, hidden_next):
        return torch.sigmoid(self.predict_layer(hidden_next).squeeze(-1))

    def _select_next_prediction(self, all_predictions, next_question):
        # Map external -1 padding to the extra all-zero row in one_hot_q.
        safe_next_question = torch.where(
            next_question != -1,
            next_question,
            torch.full_like(next_question, self.num_c),
        )
        next_one_hot = F.embedding(safe_next_question.long(), self.one_hot_q)
        return (all_predictions * next_one_hot).sum(dim=1)

    def forward(self, questions, responses):
        if questions.shape != responses.shape:
            raise ValueError("questions and responses must have identical shapes.")

        interactions = questions * self.res_len + responses
        batch_size, sequence_length = questions.shape
        hidden_t = torch.zeros(
            batch_size,
            self.num_c,
            self.hidden_dim,
            device=questions.device,
        )

        predictions = []
        for time_index in range(sequence_length - 1):
            question_t = questions[:, time_index]
            interaction_t = interactions[:, time_index]
            temporary_hidden = self._aggregate(
                interaction_t, question_t, hidden_t
            )
            hidden_t = self._update(temporary_hidden, hidden_t, question_t)
            all_predictions = self._predict_all_concepts(hidden_t)
            predictions.append(
                self._select_next_prediction(
                    all_predictions, questions[:, time_index + 1]
                )
            )

        return torch.stack(predictions, dim=1)


# -----------------------------------------------------------------------------
# Configuration: change only this section for another task/KC
# -----------------------------------------------------------------------------
SEED = 42
N_SPLITS = 3
SEQ_LEN = 20  # GKT is memory-heavy; keep identical across compared GKT runs.
BATCH_SIZE = 8
MAX_EPOCHS = 100
PATIENCE = 10
NUM_WORKERS = 0  # Use the main process; avoids DataLoader worker/file-descriptor exhaustion.
GRAPH_TYPE = "transition"

MODEL_NAME = "GKT"
KC_NAME = "exerciseid"  # Used only in filenames/OOF metadata.
QUESTION_COL = "KC (exerciseid)"       # Change to the actual KC column if needed.
CORRECT_COL = "eventualcorrect"
STUDENT_COL = "Anon Student Id"
TIMESTAMP_COL = "First Transaction Time"

# For the after-feedback task, normally change these three values:
# CORRECT_COL = "eventualcorrect"
# FIXED_FOLD_FILE = "data/fixed_outer_folds_af.csv"
# OUTPUT_DIR = Path("GKT_Optimized_AF")
FIXED_FOLD_FILE = "data/fixed_outer_folds_af.csv"
OUTPUT_DIR = Path("GKT_AF")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

PARAM_SEARCH_SPACE = {
    "hidden_dim": [32, 64, 128],
    "learning_rate": [1e-2, 1e-3, 1e-4],
    "dropout": [0.1, 0.3, 0.5],
}


# -----------------------------------------------------------------------------
# Reproducibility, device, logging, and atomic saving
# -----------------------------------------------------------------------------
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = device.type == "cuda"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger("gkt_experiment")
logger.setLevel(logging.INFO)
logger.handlers.clear()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(formatter)
file_handler = logging.FileHandler(OUTPUT_DIR / "run.log", mode="a")
file_handler.setFormatter(formatter)
logger.addHandler(stream_handler)
logger.addHandler(file_handler)


def atomic_csv(df, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)


def atomic_json(payload, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, default=str)
    os.replace(tmp, path)


def atomic_torch_save(payload, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, tmp)
    os.replace(tmp, path)


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def rmse_score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def safe_auc(y_true, y_pred):
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, y_pred))


def metrics_dict(y_true, y_prob):
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob > 0.5).astype(int)
    return {
        "AUC": safe_auc(y_true, y_prob),
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "RMSE": rmse_score(y_true, y_prob),
        "MAE": float(mean_absolute_error(y_true, y_prob)),
        "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "Recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
    }


# -----------------------------------------------------------------------------
# Data and fixed outer folds
# -----------------------------------------------------------------------------
# Uncomment/adapt this when running as a standalone .py file:
# logging_data = pd.read_csv("your_original_dataset.csv")
if "logging_data" not in globals():
    raise RuntimeError(
        "logging_data is not defined. Load the original dataset before running "
        "this script, or uncomment the pd.read_csv line above."
    )

# row_id must be created on the unfiltered original data for cross-model OOF joins.
logging_data = logging_data.copy().reset_index(drop=True)
if "row_id" not in logging_data.columns:
    logging_data["row_id"] = np.arange(len(logging_data), dtype=np.int64)

required_columns = {"row_id", QUESTION_COL, CORRECT_COL, STUDENT_COL}
missing_columns = required_columns.difference(logging_data.columns)
if missing_columns:
    raise KeyError(f"Missing required columns: {sorted(missing_columns)}")

fixed_fold_path = Path(FIXED_FOLD_FILE)
if fixed_fold_path.exists():
    fixed_folds = pd.read_csv(fixed_fold_path)
    required_fold_columns = {"row_id", "outer_fold"}
    if not required_fold_columns.issubset(fixed_folds.columns):
        raise ValueError(
            f"{FIXED_FOLD_FILE} must contain row_id and outer_fold columns."
        )
    if fixed_folds["row_id"].duplicated().any():
        raise ValueError(f"Duplicate row_id values found in {FIXED_FOLD_FILE}.")
    logging_data = logging_data.drop(columns=["outer_fold"], errors="ignore")
    logging_data = logging_data.merge(
        fixed_folds[["row_id", "outer_fold"]], on="row_id", how="left", validate="one_to_one"
    )
    logger.info("Loaded fixed folds from %s", FIXED_FOLD_FILE)
else:
    base_for_folds = logging_data.dropna(subset=[STUDENT_COL]).copy()
    base_for_folds["outer_fold"] = -1
    splitter = GroupKFold(n_splits=N_SPLITS)
    for fold, (_, test_idx) in enumerate(
        splitter.split(base_for_folds, groups=base_for_folds[STUDENT_COL]), start=1
    ):
        base_for_folds.iloc[
            test_idx, base_for_folds.columns.get_loc("outer_fold")
        ] = fold
    fixed_folds = base_for_folds[["row_id", "outer_fold"]].copy()
    atomic_csv(fixed_folds, fixed_fold_path)
    logging_data = logging_data.merge(
        fixed_folds, on="row_id", how="left", validate="one_to_one"
    )
    logger.info("Saved fixed folds to %s", FIXED_FOLD_FILE)

logging_model = logging_data.dropna(
    subset=[QUESTION_COL, CORRECT_COL, STUDENT_COL, "outer_fold"]
).copy()
logging_model[CORRECT_COL] = logging_model[CORRECT_COL].astype(int)
logging_model["outer_fold"] = logging_model["outer_fold"].astype(int)

if not logging_model[CORRECT_COL].isin([0, 1]).all():
    raise ValueError(f"{CORRECT_COL} must contain only binary values 0 and 1.")

observed_folds = set(logging_model["outer_fold"].unique())
expected_folds = set(range(1, N_SPLITS + 1))
if observed_folds != expected_folds:
    raise ValueError(
        f"Expected outer folds {sorted(expected_folds)}, found {sorted(observed_folds)}."
    )

# Every student must belong to exactly one outer fold.
student_fold_counts = logging_model.groupby(STUDENT_COL)["outer_fold"].nunique()
if (student_fold_counts > 1).any():
    raise ValueError("At least one student occurs in more than one outer fold.")

# Stable chronological order for sequences and transition edges.
sort_columns = [STUDENT_COL]
if TIMESTAMP_COL in logging_model.columns:
    sort_columns.append(TIMESTAMP_COL)
else:
    logger.warning(
        "Timestamp column %r was not found; row_id order will be used within students.",
        TIMESTAMP_COL,
    )
sort_columns.append("row_id")
logging_model = logging_model.sort_values(sort_columns, kind="mergesort").reset_index(drop=True)

# Native pyKT GKT convention: real concepts are 0..N-1 and padding is -1.
all_qids = logging_model[QUESTION_COL].drop_duplicates().tolist()
qid_to_index = {qid: index for index, qid in enumerate(all_qids)}
num_questions = len(qid_to_index)
if num_questions < 2:
    raise ValueError("GKT requires at least two distinct concepts.")

student_id_lookup = (
    logging_model[["row_id", STUDENT_COL]]
    .drop_duplicates(subset=["row_id"])
    .set_index("row_id")[STUDENT_COL]
    .to_dict()
)

logger.info("Device: %s", device)
logger.info("Model: %s | KC: %s | target: %s", MODEL_NAME, KC_NAME, CORRECT_COL)
logger.info("seq_len: %d | num_concepts: %d", SEQ_LEN, num_questions)


# -----------------------------------------------------------------------------
# Correct graph construction: one global KC mapping, training data only
# -----------------------------------------------------------------------------
def build_transition_graph_from_df(df, concept_num, qid_mapping):
    graph = np.zeros((concept_num, concept_num), dtype=np.float32)
    ordered = df.sort_values(sort_columns, kind="mergesort")

    for _, group in ordered.groupby(STUDENT_COL, sort=False):
        sequence = group[QUESTION_COL].map(qid_mapping).dropna().astype(int).to_numpy()
        if len(sequence) < 2:
            continue
        previous = sequence[:-1]
        following = sequence[1:]
        non_self = previous != following
        np.add.at(graph, (previous[non_self], following[non_self]), 1.0)

    row_sums = graph.sum(axis=1, keepdims=True)
    np.divide(graph, row_sums, out=graph, where=row_sums != 0)
    return torch.tensor(graph, dtype=torch.float32)


def build_dense_graph(concept_num):
    graph = np.full(
        (concept_num, concept_num), 1.0 / (concept_num - 1), dtype=np.float32
    )
    np.fill_diagonal(graph, 0.0)
    return torch.tensor(graph, dtype=torch.float32)


def get_gkt_graph(df):
    if GRAPH_TYPE == "transition":
        return build_transition_graph_from_df(df, num_questions, qid_to_index)
    if GRAPH_TYPE == "dense":
        return build_dense_graph(num_questions)
    raise ValueError("GRAPH_TYPE must be 'transition' or 'dense'.")


# -----------------------------------------------------------------------------
# Dataset: keeps row_id so every prediction can be aligned across models
# -----------------------------------------------------------------------------
class KTDataFromLogging(Dataset):
    def __init__(self, df):
        self.samples = []
        ordered = df.sort_values(sort_columns, kind="mergesort").copy()
        ordered["qid_index"] = ordered[QUESTION_COL].map(qid_to_index)
        if ordered["qid_index"].isna().any():
            raise ValueError("A question could not be mapped to the global KC index.")
        ordered["qid_index"] = ordered["qid_index"].astype(int)

        for _, group in ordered.groupby(STUDENT_COL, sort=False):
            q_seq = group["qid_index"].tolist()
            r_seq = group[CORRECT_COL].astype(int).tolist()
            row_seq = group["row_id"].astype(int).tolist()

            # Adjacent windows share one interaction so every target retains context.
            for start in range(0, len(q_seq), SEQ_LEN - 1):
                end = min(start + SEQ_LEN, len(q_seq))

                if end - start < 2:
                    break

                q_chunk = q_seq[start:end]
                r_chunk = r_seq[start:end]
                row_chunk = row_seq[start:end]
                pad_len = SEQ_LEN - len(q_chunk)
                if pad_len:
                    q_chunk += [-1] * pad_len
                    r_chunk += [-1] * pad_len
                    row_chunk += [-1] * pad_len
                self.samples.append(
                    (
                        torch.tensor(q_chunk, dtype=torch.long),
                        torch.tensor(r_chunk, dtype=torch.long),
                        torch.tensor(row_chunk, dtype=torch.long),
                    )
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]


def make_loader(df, shuffle):
    """
    Build a deterministic in-memory DataLoader.

    This dataset is already materialized as tensors in KTDataFromLogging, so
    multiprocessing workers are unnecessary here. Keeping NUM_WORKERS=0 also
    avoids accumulating worker pipes/epoll file descriptors across the many
    short-lived loaders created during nested CV/grid search.
    """
    loader_kwargs = {
        "dataset": KTDataFromLogging(df),
        "batch_size": BATCH_SIZE,
        "shuffle": shuffle,
        "num_workers": NUM_WORKERS,
        "pin_memory": pin_memory,
    }

    # Only enable worker persistence if multiprocessing is explicitly turned
    # back on in a future run. With NUM_WORKERS=0 no worker processes exist.
    if NUM_WORKERS > 0:
        loader_kwargs["persistent_workers"] = False

    return DataLoader(**loader_kwargs)


def make_model(params, graph):
    return GKT(
        num_c=num_questions,
        hidden_dim=params["hidden_dim"],
        emb_size=params["hidden_dim"],
        graph_type=GRAPH_TYPE,
        graph=graph,
        dropout=params["dropout"],
        emb_type="qid",
    ).to(device)


criterion = nn.BCELoss()


def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    batches = 0
    for q_batch, r_batch, _ in loader:
        q_batch = q_batch.to(device, non_blocking=True)
        r_batch = r_batch.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(q_batch, r_batch)  # [batch, seq_len - 1]
        mask = q_batch[:, 1:] != -1
        if not mask.any():
            continue
        loss = criterion(outputs[mask], r_batch[:, 1:].float()[mask])
        loss.backward()
        optimizer.step()
        total_loss += float(loss.item())
        batches += 1
    return total_loss / max(batches, 1)


def predict(model, loader, include_row_ids=False):
    model.eval()
    predictions, labels, row_ids = [], [], []
    with torch.no_grad():
        for q_batch, r_batch, rowid_batch in loader:
            q_batch = q_batch.to(device, non_blocking=True)
            r_batch = r_batch.to(device, non_blocking=True)
            outputs = model(q_batch, r_batch)
            mask = q_batch[:, 1:] != -1
            predictions.extend(outputs[mask].detach().cpu().numpy().tolist())
            labels.extend(r_batch[:, 1:][mask].detach().cpu().numpy().tolist())
            if include_row_ids:
                row_ids.extend(rowid_batch[:, 1:][mask.cpu()].numpy().tolist())
    if include_row_ids:
        return predictions, labels, row_ids
    return predictions, labels


# -----------------------------------------------------------------------------
# Outer CV
# -----------------------------------------------------------------------------
all_fold_frames = []
fold_metric_rows = []
grid_size = int(np.prod([len(values) for values in PARAM_SEARCH_SPACE.values()]))

config_payload = {
    "seed": SEED,
    "n_splits": N_SPLITS,
    "seq_len": SEQ_LEN,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "graph_type": GRAPH_TYPE,
    "model_name": MODEL_NAME,
    "kc_name": KC_NAME,
    "question_col": QUESTION_COL,
    "correct_col": CORRECT_COL,
    "student_col": STUDENT_COL,
    "timestamp_col": TIMESTAMP_COL,
    "fixed_fold_file": FIXED_FOLD_FILE,
    "param_search_space": PARAM_SEARCH_SPACE,
    "num_questions": num_questions,
    # Intentionally matches the legacy protocol used by the comparison models.
    "outer_test_used_during_training": False,
    "epoch_selection": "median_inner_best_epoch",
}
atomic_json(config_payload, OUTPUT_DIR / "run_config.json")

for fold in range(1, N_SPLITS + 1):
    fold_oof_path = OUTPUT_DIR / f"oof_{MODEL_NAME}_{KC_NAME}_fold{fold}.csv"
    fold_result_path = CHECKPOINT_DIR / f"fold_{fold}_results.csv"
    fold_model_path = CHECKPOINT_DIR / f"best_model_fold_{fold}.pth"
    training_state_path = CHECKPOINT_DIR / f"training_state_fold_{fold}.pth"

    # The result CSV is written last and acts as the fold-completion marker.
    if fold_result_path.exists() and fold_oof_path.exists() and fold_model_path.exists():
        logger.info("Fold %d is complete; loading saved OOF and metrics.", fold)
        saved_oof = pd.read_csv(fold_oof_path)
        saved_result = pd.read_csv(fold_result_path)
        all_fold_frames.append(saved_oof)
        fold_metric_rows.append(saved_result.iloc[0].to_dict())
        continue

    logger.info("=== Outer Fold %d/%d ===", fold, N_SPLITS)
    train_val_df = logging_model[logging_model["outer_fold"] != fold].copy()
    test_df = logging_model[logging_model["outer_fold"] == fold].copy()

    overlap = set(train_val_df[STUDENT_COL]).intersection(test_df[STUDENT_COL])
    if overlap:
        raise RuntimeError(f"Student leakage detected in outer fold {fold}.")

    inner_cv = GroupKFold(n_splits=N_SPLITS)
    inner_groups = train_val_df[STUDENT_COL]

    def objective(trial):
        params = {
            "hidden_dim": trial.suggest_categorical(
                "hidden_dim", PARAM_SEARCH_SPACE["hidden_dim"]
            ),
            "learning_rate": trial.suggest_categorical(
                "learning_rate", PARAM_SEARCH_SPACE["learning_rate"]
            ),
            "dropout": trial.suggest_categorical(
                "dropout", PARAM_SEARCH_SPACE["dropout"]
            ),
        }
        inner_aucs = []
        best_epochs = []

        try:
            for inner_fold, (inner_train_idx, inner_val_idx) in enumerate(
                inner_cv.split(train_val_df, groups=inner_groups), start=1
            ):
                inner_train_df = train_val_df.iloc[inner_train_idx].copy()
                inner_val_df = train_val_df.iloc[inner_val_idx].copy()

                # Crucial: the graph sees inner-training data only.
                inner_graph = get_gkt_graph(inner_train_df).to(device)
                train_loader = make_loader(inner_train_df, shuffle=True)
                val_loader = make_loader(inner_val_df, shuffle=False)
                model = make_model(params, inner_graph)
                optimizer = torch.optim.Adam(
                    model.parameters(), lr=params["learning_rate"]
                )

                best_auc = -np.inf
                best_epoch = 1
                no_improve = 0
                for epoch in range(1, MAX_EPOCHS + 1):
                    train_one_epoch(model, train_loader, optimizer)
                    val_preds, val_labels = predict(model, val_loader)
                    val_auc = safe_auc(val_labels, val_preds)
                    if np.isfinite(val_auc) and val_auc > best_auc:
                        best_auc = val_auc
                        best_epoch = epoch
                        no_improve = 0
                    else:
                        no_improve += 1
                    if no_improve >= PATIENCE:
                        break

                if not np.isfinite(best_auc):
                    raise RuntimeError(
                        f"AUC is undefined in outer fold {fold}, inner fold {inner_fold}."
                    )
                inner_aucs.append(float(best_auc))
                best_epochs.append(int(best_epoch))

                del model, optimizer, train_loader, val_loader, inner_graph
                clear_memory()

        except (torch.cuda.OutOfMemoryError, MemoryError) as error:
            logger.warning("Pruning OOM trial with parameters %s", params)
            clear_memory()
            raise optuna.TrialPruned("Out of memory") from error

        recommended_epochs = max(1, int(np.median(best_epochs)))
        trial.set_user_attr("recommended_epochs", recommended_epochs)
        trial.set_user_attr("inner_best_epochs", best_epochs)
        return float(np.mean(inner_aucs))

    storage_path = (CHECKPOINT_DIR / f"optuna_gkt_{KC_NAME}_fold{fold}.db").resolve()
    storage_url = f"sqlite:///{storage_path}"
    study = optuna.create_study(
        study_name=f"{MODEL_NAME}_{KC_NAME}_{CORRECT_COL}_fold{fold}",
        storage=storage_url,
        load_if_exists=True,
        direction="maximize",
        sampler=GridSampler(PARAM_SEARCH_SPACE, seed=SEED),
    )

    # ------------------------------------------------------------------
    # Resume the exhaustive grid safely.
    #
    # GridSampler records failed grid IDs as visited. Therefore, if a trial
    # failed because of an infrastructure problem (for example a DataLoader
    # worker/file-descriptor failure), simply reloading the study can leave
    # that parameter combination without a valid objective value.
    #
    # Re-enqueue any FAILED parameter combination that does not already have
    # a COMPLETE evaluation. This preserves all completed tuning work while
    # ensuring the failed grid point is actually evaluated.
    # ------------------------------------------------------------------
    parameter_names = tuple(PARAM_SEARCH_SPACE.keys())

    def params_key(params):
        if not all(name in params for name in parameter_names):
            return None
        return tuple(params[name] for name in parameter_names)

    complete_keys = {
        params_key(trial.params)
        for trial in study.trials
        if trial.state == optuna.trial.TrialState.COMPLETE
        and params_key(trial.params) is not None
    }

    queued_or_running_keys = {
        params_key(trial.params)
        for trial in study.trials
        if trial.state in {
            optuna.trial.TrialState.WAITING,
            optuna.trial.TrialState.RUNNING,
        }
        and params_key(trial.params) is not None
    }

    for failed_trial in study.trials:
        if failed_trial.state != optuna.trial.TrialState.FAIL:
            continue

        failed_key = params_key(failed_trial.params)
        if failed_key is None:
            continue

        if failed_key in complete_keys or failed_key in queued_or_running_keys:
            continue

        retry_params = {
            name: failed_trial.params[name]
            for name in parameter_names
        }
        study.enqueue_trial(
            retry_params,
            user_attrs={"retry_of_failed_trial": int(failed_trial.number)},
        )
        queued_or_running_keys.add(failed_key)
        logger.info(
            "Re-enqueued failed trial %d with params %s.",
            failed_trial.number,
            retry_params,
        )

    # Count unique grid combinations that already have a usable terminal
    # result. PRUNED trials retain the original behavior and count as done.
    finished_states = {
        optuna.trial.TrialState.COMPLETE,
        optuna.trial.TrialState.PRUNED,
    }
    finished_keys = {
        params_key(trial.params)
        for trial in study.trials
        if trial.state in finished_states
        and params_key(trial.params) is not None
    }

    remaining_trials = max(0, grid_size - len(finished_keys))

    logger.info(
        "Fold %d tuning resume: %d/%d unique grid combinations finished; "
        "%d trial(s) still required.",
        fold,
        len(finished_keys),
        grid_size,
        remaining_trials,
    )

    if remaining_trials:
        study.optimize(
            objective,
            n_trials=remaining_trials,
            show_progress_bar=True,
            gc_after_trial=True,
        )

    best_trial = study.best_trial
    best_params = dict(best_trial.params)
    logger.info(
        "Fold %d best params: %s | tuning AUC: %.6f",
        fold,
        best_params,
        best_trial.value,
    )

    # LEGACY COMPARABILITY PROTOCOL:
    # Retrain on all outer-training students for the epoch count selected
    # exclusively from the inner validation folds.
    final_graph = get_gkt_graph(train_val_df).to(device)
    train_loader = make_loader(train_val_df, shuffle=True)
    test_loader = make_loader(test_df, shuffle=False)
    best_model = make_model(best_params, final_graph)
    optimizer = torch.optim.Adam(
        best_model.parameters(), lr=best_params["learning_rate"]
    )

    final_epochs = int(best_trial.user_attrs["recommended_epochs"])
    start_epoch = 1
    completed_epoch = 0

    if training_state_path.exists():
        recovery = torch.load(
            training_state_path, map_location=device, weights_only=False
        )
        if (
            recovery.get("best_params") == best_params
            and int(recovery.get("final_epochs", -1)) == final_epochs
        ):
            best_model.load_state_dict(recovery["model_state_dict"])
            optimizer.load_state_dict(recovery["optimizer_state_dict"])
            start_epoch = int(recovery["completed_epoch"]) + 1
            completed_epoch = int(recovery["completed_epoch"])
            random.setstate(recovery["python_rng_state"])
            np.random.set_state(recovery["numpy_rng_state"])
            torch.set_rng_state(recovery["torch_rng_state"])
            if torch.cuda.is_available() and recovery.get("cuda_rng_state_all") is not None:
                torch.cuda.set_rng_state_all(recovery["cuda_rng_state_all"])
            logger.info("Resuming fold %d final training at epoch %d.", fold, start_epoch)

    for epoch in range(start_epoch, final_epochs + 1):
        epoch_loss = train_one_epoch(best_model, train_loader, optimizer)
        completed_epoch = epoch
        atomic_torch_save(
            {
                "completed_epoch": epoch,
                "final_epochs": final_epochs,
                "best_params": best_params,
                "model_state_dict": best_model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "python_rng_state": random.getstate(),
                "numpy_rng_state": np.random.get_state(),
                "torch_rng_state": torch.get_rng_state(),
                "cuda_rng_state_all": (
                    torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
                ),
            },
            training_state_path,
        )
        logger.info(
            "Fold %d final training epoch %d/%d | loss %.6f",
            fold, epoch, final_epochs, epoch_loss,
        )

    # Evaluate the untouched outer test fold once.
    fold_preds, fold_labels, fold_row_ids = predict(
        best_model, test_loader, include_row_ids=True
    )
    if not fold_labels:
        raise RuntimeError(f"No valid test predictions were produced for fold {fold}.")
    if len(fold_row_ids) != len(set(fold_row_ids)):
        raise RuntimeError(f"Duplicate row_id predictions found in fold {fold}.")

    fold_metrics = metrics_dict(fold_labels, fold_preds)
    logger.info(
        "Fold %d test | AUC %.6f | Accuracy %.6f | RMSE %.6f | MAE %.6f | "
        "Precision %.6f | Recall %.6f | F1 %.6f",
        fold,
        fold_metrics["AUC"],
        fold_metrics["Accuracy"],
        fold_metrics["RMSE"],
        fold_metrics["MAE"],
        fold_metrics["Precision"],
        fold_metrics["Recall"],
        fold_metrics["F1"],
    )

    fold_df = pd.DataFrame(
        {
            "row_id": fold_row_ids,
            "student_id": [student_id_lookup[row_id] for row_id in fold_row_ids],
            "fold": fold,
            "y_true": fold_labels,
            "y_pred": fold_preds,
            "model_name": MODEL_NAME,
            "kc_name": KC_NAME,
        }
    ).sort_values("row_id").reset_index(drop=True)

    result_row = {
        "Fold": fold,
        "Tuning_AUC": float(best_trial.value),
        **fold_metrics,
        "Final_Epochs": final_epochs,
        "Best_Params": json.dumps(best_params, sort_keys=True),
    }

    # Crash-safe order: OOF -> model -> completion marker (result CSV).
    atomic_csv(fold_df, fold_oof_path)
    atomic_torch_save(
        {
            "model_state_dict": best_model.state_dict(),
            "best_params": best_params,
            "final_epochs": final_epochs,
            "qid_to_index": qid_to_index,
            "config": config_payload,
        },
        fold_model_path,
    )
    atomic_csv(pd.DataFrame([result_row]), fold_result_path)
    logger.info("Saved fold %d model, metrics, and OOF predictions.", fold)

    all_fold_frames.append(fold_df)
    fold_metric_rows.append(result_row)
    del best_model, optimizer, train_loader, test_loader, final_graph
    clear_memory()


# -----------------------------------------------------------------------------
# Combined OOF and final pooled/fold-averaged results (also works after resume)
# -----------------------------------------------------------------------------
if len(all_fold_frames) != N_SPLITS:
    raise RuntimeError(
        f"Expected {N_SPLITS} completed fold OOF files, found {len(all_fold_frames)}."
    )

oof_df = pd.concat(all_fold_frames, ignore_index=True).sort_values("row_id").reset_index(drop=True)
if oof_df["row_id"].duplicated().any():
    raise RuntimeError("Duplicate row_id values found across combined OOF predictions.")

combined_oof_path = OUTPUT_DIR / f"oof_{MODEL_NAME}_{KC_NAME}_all.csv"
atomic_csv(oof_df, combined_oof_path)

fold_results_df = pd.DataFrame(fold_metric_rows).sort_values("Fold").reset_index(drop=True)
atomic_csv(fold_results_df, OUTPUT_DIR / "fold_results_all.csv")

metric_names = ["AUC", "Accuracy", "RMSE", "MAE", "Precision", "Recall", "F1"]
pooled = metrics_dict(oof_df["y_true"], oof_df["y_pred"])
summary_rows = []
for metric in metric_names:
    values = pd.to_numeric(fold_results_df[metric], errors="coerce").to_numpy(dtype=float)
    values = values[np.isfinite(values)]
    summary_rows.append(
        {
            "Metric": "F1 Score" if metric == "F1" else metric,
            "Pooled Score": pooled[metric],
            "Average Score": float(np.mean(values)),
            "Std": float(np.std(values, ddof=1)) if len(values) > 1 else np.nan,
        }
    )

summary_df = pd.DataFrame(summary_rows)
atomic_csv(summary_df, OUTPUT_DIR / "final_metrics.csv")

logger.info("Saved combined OOF predictions to %s", combined_oof_path)
logger.info("=== Final Evaluation Across All Folds ===")
logger.info("Metric       | Pooled Score | Average Score ± Std")
logger.info("-------------|--------------|---------------------")
for row in summary_rows:
    logger.info(
        "%-12s | %.6f     | %.6f ± %.6f",
        row["Metric"],
        row["Pooled Score"],
        row["Average Score"],
        row["Std"],
    )